# Focal Variety

Focal variety counts the number of distinct values in a sliding
neighbourhood window.  It is most useful for categorical rasters
(land-cover, soil type, geology codes) where you want to map
boundary complexity or patch fragmentation.

This notebook shows how to compute focal variety with
`xrspatial.focal.focal_stats` across different kernel shapes.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial.convolution import circle_kernel, custom_kernel
from xrspatial.focal import focal_stats

## Create a synthetic land-cover raster

We build a 60x60 grid with four land-cover classes arranged in
quadrants, plus a few scattered patches to make things interesting.

In [ ]:
rng = np.random.default_rng(42)
rows, cols = 60, 60

# Four quadrants: classes 1-4
lc = np.ones((rows, cols), dtype=np.float64)
lc[:rows//2, cols//2:] = 2
lc[rows//2:, :cols//2] = 3
lc[rows//2:, cols//2:] = 4

# Scatter some class-5 patches
for _ in range(30):
    r, c = rng.integers(0, rows), rng.integers(0, cols)
    lc[r:r+3, c:c+3] = 5

land_cover = xr.DataArray(lc, dims=['y', 'x'], name='land_cover')

fig, ax = plt.subplots(figsize=(5, 5))
land_cover.plot(ax=ax, cmap='Set2', add_colorbar=True)
ax.set_title('Synthetic land-cover raster')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Compute focal variety with a 3x3 box kernel

A 3x3 box kernel counts how many distinct classes appear in the
immediate 8-connected neighbourhood of each pixel.

In [ ]:
kernel_box = np.ones((3, 3))
result_box = focal_stats(land_cover, kernel_box, stats_funcs=['variety'])
variety_box = result_box.sel(stats='variety')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
land_cover.plot(ax=axes[0], cmap='Set2', add_colorbar=True)
axes[0].set_title('Land cover')
axes[0].set_aspect('equal')

variety_box.plot(ax=axes[1], cmap='YlOrRd', add_colorbar=True)
axes[1].set_title('Focal variety (3x3 box)')
axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

Pixels deep inside a uniform quadrant show variety = 1.  Pixels on
boundaries between classes show variety = 2, 3, or 4 depending on
how many classes meet at that point.  The scattered class-5 patches
create small pockets of higher variety.

## Larger kernel: 5x5 circle

Increasing the kernel radius captures more of the surrounding
landscape, so variety values near boundaries will be higher.

In [ ]:
kernel_circle = circle_kernel(2, 2, 2)
result_circle = focal_stats(land_cover, kernel_circle, stats_funcs=['variety'])
variety_circle = result_circle.sel(stats='variety')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
variety_box.plot(ax=axes[0], cmap='YlOrRd', add_colorbar=True)
axes[0].set_title('Variety (3x3 box)')
axes[0].set_aspect('equal')

variety_circle.plot(ax=axes[1], cmap='YlOrRd', add_colorbar=True)
axes[1].set_title('Variety (5x5 circle)')
axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

## Combining variety with other focal stats

You can request variety alongside other statistics in one call.
Here we grab both range and variety to compare continuous and
categorical measures of local heterogeneity.

In [ ]:
result_combo = focal_stats(land_cover, kernel_box,
                           stats_funcs=['range', 'variety'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
result_combo.sel(stats='range').plot(ax=axes[0], cmap='viridis',
                                     add_colorbar=True)
axes[0].set_title('Focal range')
axes[0].set_aspect('equal')

result_combo.sel(stats='variety').plot(ax=axes[1], cmap='YlOrRd',
                                       add_colorbar=True)
axes[1].set_title('Focal variety')
axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

Range measures the numeric spread (max minus min) while variety
counts distinct classes.  For categorical data, variety is usually
the more meaningful measure since the numeric distance between
class codes is arbitrary.